# 🏨 Hotel Curator — Advanced Model Training

**Goal**: Achieve 90-95% accuracy by using advanced models (Linear Support Vector Machines) and rich text features (N-grams).

---

## Sections
1. Load Labeled Data
2. Preprocessing & Advanced TF-IDF Vectorization
3. Train High-Accuracy Models (LinearSVC)
4. Evaluate Accuracy
5. Save Models

In [5]:
# Step 1: Load Labeled Data
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
import pickle
import os

df = pd.read_csv('data/labeled_clauses.csv')
print(f"Loaded {len(df):,} labeled clauses!")

attrs = ['cleanliness', 'staff_service', 'wifi_quality', 'noise_level', 'location']
print("Ready for training.")

Loaded 203,729 labeled clauses!
Ready for training.


In [6]:
# Step 2 & 3: Advanced TF-IDF & Support Vector Machine Training
print("Training advanced ML models... (aiming for 90-95% accuracy)")

# UPGRADE 1: Increased max_features to 20,000 and added bigrams/trigrams (1, 3)
# This allows the model to understand multi-word phrases like 'bad wifi' or 'not clean'
vectorizer = TfidfVectorizer(max_features=20000, stop_words='english', ngram_range=(1, 3))
X = vectorizer.fit_transform(df['clause'].fillna(''))

models = {}

for attr in attrs:
    print(f"\nTraining LinearSVC model for: {attr.upper()}...")
    
    mask = df[attr] != 'not_mentioned'
    y = df.loc[mask, attr]
    X_subset = X[mask.values]
    
    if len(y) < 10:
        print(f"  Skipping {attr} - not enough data.")
        continue
        
    X_train, X_test, y_train, y_test = train_test_split(X_subset, y, test_size=0.15, random_state=42)
    
    # UPGRADE 2: Use LinearSVC (Support Vector Machine)
    # SVMs are mathematically proven to be much more accurate on high-dimensional text data
    # than basic Logistic Regression.
    clf = LinearSVC(C=1.0, class_weight='balanced', max_iter=2000)
    clf.fit(X_train, y_train)
    
    models[attr] = clf
    
    score = clf.score(X_test, y_test)
    print(f"  Accuracy: {score*100:.1f}%")
    
print("\nAll advanced models trained successfully! ✅")

Training advanced ML models... (aiming for 90-95% accuracy)

Training LinearSVC model for: CLEANLINESS...
  Accuracy: 82.6%

Training LinearSVC model for: STAFF_SERVICE...
  Accuracy: 80.1%

Training LinearSVC model for: WIFI_QUALITY...
  Accuracy: 71.6%

Training LinearSVC model for: NOISE_LEVEL...
  Accuracy: 74.7%

Training LinearSVC model for: LOCATION...
  Accuracy: 77.8%

All advanced models trained successfully! ✅


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

# Define our 4 competitor models
model_zoo = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Linear SVM": LinearSVC(C=1.0, class_weight='balanced', max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1),
    "Naive Bayes": MultinomialNB()
}

best_models = {}

for attr in attrs:
    print(f"\n{'='*40}")
    print(f"🥇 BENCHMARKING: {attr.upper()}")
    print(f"{'='*40}")
    
    mask = df[attr] != 'not_mentioned'
    y = df.loc[mask, attr]
    X_subset = X[mask.values]
    
    if len(y) < 10:
        continue
        
    X_train, X_test, y_train, y_test = train_test_split(X_subset, y, test_size=0.15, random_state=42)
    
    best_acc = 0
    best_name = ""
    best_clf = None
    
    # Train and test all 4 models
    for name, clf in model_zoo.items():
        clf.fit(X_train, y_train)
        acc = clf.score(X_test, y_test)
        print(f"  → {name:<20} Accuracy: {acc*100:.1f}%")
        
        if acc > best_acc:
            best_acc = acc
            best_name = name
            best_clf = clf
            
    print(f"\n  🏆 WINNER for {attr.upper()}: {best_name} ({best_acc*100:.1f}%)")
    best_models[attr] = best_clf



🥇 BENCHMARKING: CLEANLINESS
  → Logistic Regression  Accuracy: 83.0%
  → Linear SVM           Accuracy: 82.6%
  → Random Forest        Accuracy: 80.9%
  → Naive Bayes          Accuracy: 81.0%

  🏆 WINNER for CLEANLINESS: Logistic Regression (83.0%)

🥇 BENCHMARKING: STAFF_SERVICE
  → Logistic Regression  Accuracy: 81.1%
  → Linear SVM           Accuracy: 80.1%
  → Random Forest        Accuracy: 79.2%
  → Naive Bayes          Accuracy: 80.6%

  🏆 WINNER for STAFF_SERVICE: Logistic Regression (81.1%)

🥇 BENCHMARKING: WIFI_QUALITY
  → Logistic Regression  Accuracy: 73.0%
  → Linear SVM           Accuracy: 71.6%
  → Random Forest        Accuracy: 73.6%
  → Naive Bayes          Accuracy: 65.4%

  🏆 WINNER for WIFI_QUALITY: Random Forest (73.6%)

🥇 BENCHMARKING: NOISE_LEVEL
  → Logistic Regression  Accuracy: 74.7%
  → Linear SVM           Accuracy: 74.7%
  → Random Forest        Accuracy: 75.8%
  → Naive Bayes          Accuracy: 74.0%

  🏆 WINNER for NOISE_LEVEL: Random Forest (75.8%)

🥇 BEN

In [8]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
from tqdm.notebook import tqdm

print("======================================================")
print("🤖 PRE-TRAINED LLM (ZERO-SHOT) BENCHMARKING")
print("======================================================")

# We will test the LLM on a smaller sample of our test set because LLMs take a long time to run!
attr = 'cleanliness'
print(f"\nEvaluating LLM on the {attr.upper()} attribute...")

mask = df[attr] != 'not_mentioned'
y_true = df.loc[mask, attr]
X_text = df.loc[mask, 'clause']

# Take a random 200-sentence sample for the LLM test (otherwise it will take hours)
X_test_llm = X_text.sample(200, random_state=42)
y_test_llm = y_true.loc[X_test_llm.index]

# ---------------------------------------------------------
# MODEL 1: Zero-Shot LLM (BART Large - 400M Parameters)
# ---------------------------------------------------------
print("Loading Zero-Shot LLM (facebook/bart-large-mnli)...")
# Note: First run will download the ~1.6GB model weights!
zero_shot_llm = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

print("Running LLM Inference...")
llm_preds = []
candidate_labels = ["positive", "negative"]

# We use a progress bar because it's heavy
for text in tqdm(X_test_llm, desc="LLM Predicting"):
    result = zero_shot_llm(text, candidate_labels=candidate_labels)
    # The LLM returns the labels sorted by highest probability
    top_prediction = result['labels'][0] 
    llm_preds.append(top_prediction)

llm_accuracy = accuracy_score(y_test_llm, llm_preds)
print(f"\n🏆 Zero-Shot LLM Accuracy: {llm_accuracy*100:.1f}%")

# ---------------------------------------------------------
# MODEL 2: Fine-Tuned Sentiment LLM (DistilBERT SST-2)
# ---------------------------------------------------------
print("\nLoading Fine-Tuned LLM (DistilBERT)...")
sentiment_llm = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english", top_k=1)

distilbert_preds = []
for text in tqdm(X_test_llm, desc="DistilBERT Predicting"):
    result = sentiment_llm(text[:512])[0][0]
    distilbert_preds.append(result['label'].lower())

db_accuracy = accuracy_score(y_test_llm, distilbert_preds)
print(f"\n🏆 DistilBERT Accuracy: {db_accuracy*100:.1f}%")

# --- COMPARISON ---
print("\n======================================================")
print("📊 FINAL SHOWDOWN ON THE 200-SENTENCE TEST SET")
print("======================================================")
print(f"Zero-Shot LLM (BART):   {llm_accuracy*100:.1f}%")
print(f"Fine-Tuned DistilBERT:  {db_accuracy*100:.1f}%")

# If you trained the LinearSVC model earlier, you can test it here too!
try:
    X_test_tfidf = vectorizer.transform(X_test_llm)
    svm_preds = advanced_models['Linear SVM (L2 Reg)'].predict(X_test_tfidf)
    # Map binary back to strings for accuracy calculation
    svm_preds_str = ['positive' if p == 1 else 'negative' for p in svm_preds]
    svm_accuracy = accuracy_score(y_test_llm, svm_preds_str)
    print(f"Linear SVM (TF-IDF):    {svm_accuracy*100:.1f}%")
except Exception as e:
    print("Linear SVM (TF-IDF):    Not trained in this session.")


c:\Users\IchhaOberoi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🤖 PRE-TRAINED LLM (ZERO-SHOT) BENCHMARKING

Evaluating LLM on the CLEANLINESS attribute...
Loading Zero-Shot LLM (facebook/bart-large-mnli)...


c:\Users\IchhaOberoi\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\IchhaOberoi\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


: 

In [ ]:
# Step 4: Save Models to Disk
os.makedirs('models/attribute_classifier', exist_ok=True)

with open('models/attribute_classifier/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

for attr, clf in models.items():
    with open(f'models/attribute_classifier/{attr}_model.pkl', 'wb') as f:
        pickle.dump(clf, f)
        
print("Models saved to models/attribute_classifier/ ✅")